# ⚡ Notebook 1: Circuit Breaker — from Bad to Best

When a downstream service is sick, pounding it with more requests **makes things worse**:
- the sick service stays overloaded and can't recover,
- our threads pile up waiting for timeouts,
- callers upstream of us time out too → the failure cascades.

A **circuit breaker** watches failures and, when they cross a threshold, **opens** — short-circuiting the call and returning an error immediately, without touching the downstream.

### Analogy 🏠
Your home's electrical breaker trips so the wires don't melt. Same idea — a fuse for software.

### Three states
```
                 failures cross threshold
      +--------+ ------------------------> +--------+
      | CLOSED |                           |  OPEN  |
      +--------+ <------------------------ +--------+
           ^        trial call succeeded        |
           |                                    | cool-down elapsed
           |                                    v
           |          trial call failed    +-----------+
           +------------------------------ | HALF_OPEN |
                    (back to OPEN)         +-----------+
```
1. `CLOSED` — all good, calls pass through.
2. `OPEN` — too many failures; calls fail instantly (fast fail).
3. `HALF_OPEN` — after a cool-down, allow a trial call. Success → `CLOSED`. Failure → `OPEN` again.

We'll walk **bad → better → best** implementations so you can see *why* each piece exists.

## 🛠️ Setup

```bash
cd 05-microservices/circuit-breaker
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 😱 Step 0 — Bad: no breaker

We just call the downstream. Every failure costs us the full timeout. Imagine this is a 1-second timeout across 10 requests — that's 10 seconds of blocked threads, for nothing.

In [ ]:
import time

def sick_service():
    # Pretend the downstream hangs for 0.2s, then errors out.
    time.sleep(0.2)
    raise RuntimeError('downstream down')

def call_no_breaker():
    try:
        return sick_service()
    except Exception as e:
        return f'FAIL: {e}'

t0 = time.time()
results = [call_no_breaker() for _ in range(10)]
print(f'no breaker: {time.time()-t0:.2f}s for 10 reqs — every call paid the timeout')


## 🙂 Step 1 — Naive breaker (counter only)

First try: count failures; after N in a row, **stop calling** and return an error instantly.

**Problem:** once it opens, it never recovers. The downstream could be healthy again, but we never check.


In [ ]:
class NaiveBreaker:
    def __init__(self, fail_threshold=3):
        self.failures = 0
        self.open = False
        self.fail_threshold = fail_threshold

    def call(self, fn, *a, **kw):
        if self.open:
            raise RuntimeError('circuit OPEN — failing fast')
        try:
            return fn(*a, **kw)
        except Exception:
            self.failures += 1
            if self.failures >= self.fail_threshold:
                self.open = True
            raise

cb = NaiveBreaker(fail_threshold=3)
t0 = time.time()
for i in range(10):
    try:
        cb.call(sick_service)
    except Exception as e:
        print(f'  {i}: {e}')
print(f'naive breaker: {time.time()-t0:.2f}s for 10 reqs (fast-fails after 3 failures)')
print(f'breaker stays OPEN forever? {cb.open}  ← the bug: no recovery path')


## 🙂🙂 Step 2 — Better: add `HALF_OPEN` and auto-reset

After a **cool-down**, we let *one trial call* through. If it works, the downstream probably recovered → close the circuit. If it fails, open again.

This is the **classic circuit breaker** pattern.

In [ ]:
class CircuitBreaker:
    def __init__(self, fail_threshold=3, reset_after=1.0):
        self.state = 'CLOSED'
        self.failures = 0
        self.opened_at = 0.0
        self.fail_threshold = fail_threshold
        self.reset_after = reset_after

    def call(self, fn, *a, **kw):
        # If OPEN and enough time has passed, try a single probe.
        if self.state == 'OPEN':
            if time.time() - self.opened_at >= self.reset_after:
                self.state = 'HALF_OPEN'
                print('  → HALF_OPEN (trial call allowed)')
            else:
                raise RuntimeError('circuit OPEN — failing fast')

        try:
            result = fn(*a, **kw)
        except Exception:
            self.failures += 1
            # One failed trial re-opens immediately.
            if self.state == 'HALF_OPEN' or self.failures >= self.fail_threshold:
                self.state = 'OPEN'
                self.opened_at = time.time()
                print('  → OPEN')
            raise

        # Success — close the circuit and reset counter.
        if self.state == 'HALF_OPEN':
            print('  → CLOSED (trial passed)')
        self.state = 'CLOSED'
        self.failures = 0
        return result

def flaky(fail=True):
    if fail:
        raise RuntimeError('downstream down')
    return 'ok'

cb = CircuitBreaker(fail_threshold=3, reset_after=1.0)

print('Phase 1: downstream is broken')
for i in range(6):
    try:
        cb.call(flaky, fail=True)
        print(f'  {i}: ok')
    except Exception as e:
        print(f'  {i}: {e}')

print('\n... waiting out the cool-down ...')
time.sleep(1.1)

print('\nPhase 2: downstream has recovered')
for i in range(3):
    try:
        print(f'  {i}:', cb.call(flaky, fail=False))
    except Exception as e:
        print(f'  {i}: {e}')


## 🏆 Step 3 — Best: time-windowed failures + thread safety

Two subtle real-world issues with Step 2:

1. **Counter never resets over time.** 2 failures today + 1 failure tomorrow shouldn't trip the breaker. Production breakers track failures within a **rolling time window** (e.g. last 10s).
2. **Thread safety.** In a real server many requests run in parallel. State transitions need a lock, or we can get double-opens and weird races.

Here's a tiny windowed, thread-safe version:

In [ ]:
import threading
from collections import deque

class WindowedBreaker:
    """Trips when failures in the last `window_s` seconds exceed `fail_threshold`.

    Three things this adds over Step 2:
      1. a rolling time window, so ancient failures are forgotten;
      2. a lock, so concurrent requests can't race the state machine;
      3. a cap on how many trial calls may run in HALF_OPEN at once.
    """
    def __init__(self, fail_threshold=3, window_s=10.0, reset_after=1.0,
                 half_open_max_calls=1):
        self.state = 'CLOSED'
        self.failure_times = deque()      # timestamps of recent failures
        self.opened_at = 0.0
        self.fail_threshold = fail_threshold
        self.window_s = window_s
        self.reset_after = reset_after
        self.half_open_max_calls = half_open_max_calls
        self.half_open_inflight = 0
        self.lock = threading.Lock()

    def _trim(self, now):
        cutoff = now - self.window_s
        while self.failure_times and self.failure_times[0] < cutoff:
            self.failure_times.popleft()

    def call(self, fn, *a, **kw):
        probing = False
        with self.lock:
            now = time.time()
            if self.state == 'OPEN':
                if now - self.opened_at >= self.reset_after:
                    self.state = 'HALF_OPEN'
                    self.half_open_inflight = 0
                else:
                    raise RuntimeError('circuit OPEN — failing fast')
            if self.state == 'HALF_OPEN':
                # ⚠️ Without this cap, EVERY request that arrives during the
                # cool-down window is let through as a "trial" — and a fleet of
                # them lands on the service we were trying to protect.
                if self.half_open_inflight >= self.half_open_max_calls:
                    raise RuntimeError('circuit HALF_OPEN — trial already in flight')
                self.half_open_inflight += 1
                probing = True

        try:
            result = fn(*a, **kw)
        except Exception:
            with self.lock:
                now = time.time()
                if probing:
                    self.half_open_inflight -= 1
                self.failure_times.append(now)
                self._trim(now)
                if self.state == 'HALF_OPEN' or len(self.failure_times) >= self.fail_threshold:
                    self.state = 'OPEN'
                    self.opened_at = now
            raise

        with self.lock:
            if probing:
                self.half_open_inflight -= 1
            self.state = 'CLOSED'
            self.failure_times.clear()
        return result

# Demo A: 2 failures, wait out the window, 1 more failure → should NOT trip.
cb = WindowedBreaker(fail_threshold=3, window_s=2.0, reset_after=1.0)
for i in range(2):
    try: cb.call(flaky, fail=True)
    except Exception: pass
print(f'after 2 quick failures: state={cb.state}')

time.sleep(2.1)  # let old failures fall out of the window
try: cb.call(flaky, fail=True)
except Exception: pass
print(f'after window expired + 1 more failure: state={cb.state} (old failures forgotten)')


### Demo B: why HALF_OPEN needs a concurrency cap

The point of the cool-down is to send the sick service **one** careful probe, not a
fresh wave. Here 20 threads all arrive the instant the cool-down expires.
With `half_open_max_calls=1` exactly one of them is allowed to touch the downstream;
the other 19 are rejected without a network call.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

probe_hits = {'n': 0}
hits_lock = threading.Lock()

def still_broken(slow=False):
    with hits_lock:
        probe_hits['n'] += 1   # counts requests that actually reached the service
    if slow:
        time.sleep(0.3)        # a real probe takes time — that's the window for a stampede
    raise RuntimeError('downstream still down')

def stampede(half_open_max_calls):
    probe_hits['n'] = 0
    cb = WindowedBreaker(fail_threshold=3, window_s=10.0, reset_after=0.2,
                         half_open_max_calls=half_open_max_calls)
    for _ in range(3):                          # trip the breaker
        try: cb.call(still_broken)
        except Exception: pass
    tripped_at = probe_hits['n']
    time.sleep(0.25)                            # cool-down expires

    def one(_):
        try: return cb.call(still_broken, slow=True)
        except RuntimeError: return None
    with ThreadPoolExecutor(max_workers=20) as ex:
        list(ex.map(one, range(20)))
    return probe_hits['n'] - tripped_at

print(f'half_open_max_calls=1  -> {stampede(1):>2d}/20 requests reached the sick service')
print(f'no cap (=20)           -> {stampede(20):>2d}/20 requests reached the sick service')
print()
print('The uncapped version undoes the whole point of the cool-down: the service')
print('gets a burst at the exact moment it was supposed to be left alone.')

> ⚠️ **One more honest caveat about this implementation.** A single success sets the
> state back to `CLOSED` *and clears the whole failure window*. That makes it a
> "consecutive failures" breaker in disguise — a dependency failing 40% of the time
> may never accumulate 3 failures in a row and so never trips.
>
> Production breakers (Resilience4j, `gobreaker`, Envoy) instead track **failure
> *rate*** over the window: they count successes *and* failures and trip when
> `failures / total > threshold`, with a **minimum call count** so a single early
> failure can't trip a 100%-failure-rate breaker. Reach for a library rather than
> re-deriving that.

## ⏱️ The piece everyone forgets: a **timeout in front of the breaker**

A breaker counts *errors*. A dependency that is merely **slow** never produces one —
it just holds your thread. So the breaker sits there in `CLOSED`, perfectly happy,
while every worker in your process is parked on a socket read.

**A circuit breaker without a timeout is decorative.** Watch it happen:

In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FTimeout

def hangs_forever():
    time.sleep(5.0)          # a "slow" dependency: never errors, just... waits
    return 'ok'

# --- A) breaker with NO timeout in front of it ---
cb_no_timeout = WindowedBreaker(fail_threshold=3, window_s=10.0, reset_after=1.0)

def attempt_no_timeout():
    t0 = time.time()
    try:
        cb_no_timeout.call(hangs_forever)
    except Exception:
        pass
    return time.time() - t0

pool = ThreadPoolExecutor(max_workers=3)
futures = [pool.submit(attempt_no_timeout) for _ in range(3)]
time.sleep(1.0)   # let the calls get stuck
print('after 1s of a hanging dependency:')
print(f'  breaker state = {cb_no_timeout.state}   ← still CLOSED. It saw zero errors.')
print(f'  stuck workers = {sum(1 for f in futures if not f.done())}/3   ← the pool is gone')

# --- B) same dependency, but each call gets a per-attempt timeout FIRST ---
def with_timeout(fn, timeout_s):
    """In real code this is just `requests.get(url, timeout=...)`."""
    f = ThreadPoolExecutor(max_workers=1).submit(fn)
    try:
        return f.result(timeout=timeout_s)
    except FTimeout:
        raise TimeoutError(f'per-attempt timeout after {timeout_s}s')

cb_with_timeout = WindowedBreaker(fail_threshold=3, window_s=10.0, reset_after=1.0)
t0 = time.time()
for i in range(5):
    try:
        cb_with_timeout.call(with_timeout, hangs_forever, 0.1)
        print(f'  req {i}: ok')
    except Exception as e:
        print(f'  req {i}: {type(e).__name__}: {e}')
print(f'  breaker state = {cb_with_timeout.state}   '
      f'← tripped in {time.time()-t0:.2f}s, threads free')

pool.shutdown(wait=False)

Read the two outputs side by side:

- **Without a timeout** the breaker stays `CLOSED` forever and your worker pool
  drains away. The dependency never "failed", so there was nothing to count.
- **With a timeout** the slowness is *converted into an error*, the breaker sees
  three of them, trips, and every subsequent call fails in microseconds.

That is the actual composition rule: **timeout → error → breaker**. Set the
per-attempt timeout from the dependency's healthy p99 plus headroom — not from
how long you're willing to wait, which is a different (and larger) number.

## 🎯 Key settings to tune

| Parameter | What it controls | Too low | Too high |
|---|---|---|---|
| `fail_threshold` | failures before we trip | flaps on tiny blips | slow to react to outages |
| `window_s` | how far back failures count | forgets real outages | one-off errors add up over days |
| `reset_after` | cool-down before probing | hammers a recovering service | stays open longer than needed |

There is no universal answer — tune to your service's normal error rate and latency.

### What's next
Notebook 2 shows the breaker in action under concurrent load — where it actually saves your system from a cascading failure.